In [0]:
%sql
MERGE INTO bootcamp_matias.gold.fact_propiedades AS t
USING (
 SELECT * FROM ( 
  SELECT 
  B.zona_id,
  C.estado_id,
  D.tipo_operacion_id,
  A.precio,
  A.moneda,
  A.expensas,
  A.ambientes,
  A.metros_cuadrados_totales,
  A.metros_cuadrados_cubiertos,
  A.antiguedad,
  A.url,
  A.precio_por_m2,
  ROW_NUMBER() OVER (
    PARTITION BY A.url
        ORDER BY A.precio
      ) AS rn
  FROM bootcamp_matias.silver.propiedades A
  LEFT JOIN bootcamp_matias.gold.dim_zona B
    ON A.ciudad = B.ciudad
  LEFT JOIN bootcamp_matias.gold.dim_estado C
    ON A.estado = C.estado
  LEFT JOIN bootcamp_matias.gold.dim_tipo_operacion D
    ON A.tipo_operacion = D.tipo_operacion

)s
WHERE rn = 1

) src

ON t.url = src.url

WHEN MATCHED AND COALESCE(t.precio,0) <> COALESCE(src.precio,0)
THEN UPDATE SET
  t.precio = src.precio,
  t.moneda = src.moneda,
  t.expensas = src.expensas,
  t.precio_por_m2 = src.precio_por_m2

WHEN NOT MATCHED THEN
INSERT(
  zona_id,
  estado_id,
  tipo_operacion_id,
  precio,
  moneda,
  expensas,
  ambientes,
  metros_cuadrados_totales,
  metros_cuadrados_cubiertos,
  antiguedad,
  url,
  precio_por_m2,
  _refresh_timestamp
)
VALUES(
  src.zona_id,
  src.estado_id,
  src.tipo_operacion_id,
  src.precio,
  src.moneda,
  src.expensas,
  src.ambientes,
  src.metros_cuadrados_totales,
  src.metros_cuadrados_cubiertos,
  src.antiguedad,
  src.url,
  src.precio_por_m2,
  CURRENT_TIMESTAMP()
)